# ⚽ FIFA 2027 AI News Poster Generator

An end-to-end pipeline that scrapes football news, scores articles for viral
potential, generates social-media content, and composites poster images.

## Quick start

1. Ensure `.env` exists next to this notebook:
```
AI_PROVIDER=groq
AI_API_KEY=your-groq-key
AI_MODEL=llama-3.3-70b-versatile
IMAGE_BACKEND=mock
MIN_VIRAL_SCORE=5
MAX_ARTICLES_PER_RUN=20
```
2. Install dependencies:
```bash
pip install -r requirements.txt
```
3. **Run all cells top-to-bottom** (Kernel → Restart & Run All).

## Image generation
Set `IMAGE_BACKEND` in `.env`:
| Value | Backend | Key needed |
|-------|---------|------------|
| `mock` | Solid-colour placeholder (default) | — |
| `stability` | Stable Diffusion XL via Stability AI | `STABILITY_API_KEY` |
| `replicate` | Flux via Replicate | `REPLICATE_API_KEY` |

## Cells overview
| # | Name | What it does |
|---|------|--------------|
| 1 | Imports | Load env, imports |
| 2 | RSS Fetch | Scrape all news sources |
| 3 | Dedup | Remove duplicate articles |
| 4 | AI Client | Multi-provider LLM wrapper |
| 5 | Scoring | 4-dimension viral scoring |
| 6 | Content | Social captions + image prompts |
| 7 | Images | Generate background images |
| 8 | Posters | Composite final posters (3 formats) |
| 9 | Export | JSON / CSV / HTML + console report |

> **Production mode**: use `python main.py` for scheduled automated runs.


In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 – Imports and environment loading
# ─────────────────────────────────────────────────────────────────

import os, re, json, time, textwrap, io
from pathlib import Path
from datetime import datetime

import feedparser
import requests
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv

load_dotenv()

AI_PROVIDER = os.getenv("AI_PROVIDER", "groq").lower()
AI_API_KEY  = os.getenv("AI_API_KEY",  "")
AI_MODEL    = os.getenv("AI_MODEL",    "llama-3.3-70b-versatile")
IMAGE_BACKEND = os.getenv("IMAGE_BACKEND", "mock")
MIN_VIRAL_SCORE = int(os.getenv("MIN_VIRAL_SCORE", "5"))
MAX_ARTICLES    = int(os.getenv("MAX_ARTICLES_PER_RUN", "20"))

if not AI_API_KEY:
    raise EnvironmentError(
        "AI_API_KEY is not set. "
        "Create a .env file with AI_API_KEY=<your key> and re-run this cell."
    )

print(f"✅  Provider : {AI_PROVIDER}")
print(f"✅  Model    : {AI_MODEL}")
print(f"✅  Image    : {IMAGE_BACKEND}")
print("✅  API key  : ***hidden***")

✅  Provider : groq
✅  Model    : llama-3.3-70b-versatile
✅  Image    : gemini
✅  API key  : ***hidden***


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 – RSS feed fetcher + HTML scraping fallback
#
# Fetches from all FIFA 2027 news sources.
# Normalises every article into the same dict schema.
# ─────────────────────────────────────────────────────────────────

RSS_SOURCES = {
    "FIFA News":     "https://www.fifa.com/rss-feeds/news/en.xml",
    "BBC Sport":     "https://feeds.bbci.co.uk/sport/football/rss.xml",
    "Sky Sports":    "https://www.skysports.com/rss/11095",
    "ESPN Soccer":   "https://www.espn.com/espn/rss/soccer/news",
    "Goal.com":      "https://www.goal.com/feeds/en/news",
    "Reuters Sport": "https://feeds.reuters.com/reuters/sportsNews",
}

_HEADERS = {"User-Agent": "FIFA2027Bot/1.0"}


def _clean_html(text):
    return re.sub(r"<[^>]+>", "", text or "").strip()


def fetch_rss_articles(sources):
    """Fetch all RSS feeds; return flat list of normalised article dicts."""
    all_articles = []

    for source_name, feed_url in sources.items():
        print(f"Fetching {source_name} ...", end=" ")
        try:
            feed = feedparser.parse(feed_url, request_headers=_HEADERS)
            if feed.bozo:
                print("(parse warning)", end=" ")
            count = 0
            for entry in feed.entries:
                summary  = _clean_html(getattr(entry, "summary", "") or "")
                content  = ""
                if hasattr(entry, "content") and entry.content:
                    content = _clean_html(entry.content[0].get("value", ""))
                all_articles.append({
                    "title":     (entry.get("title") or "No title").strip(),
                    "summary":   summary,
                    "full_text": content or summary,
                    "link":      entry.get("link", ""),
                    "source":    source_name,
                    "published": getattr(entry, "published", ""),
                })
                count += 1
            print(f"OK ({count} articles)")
        except Exception as exc:
            print(f"FAILED – {exc}")
        time.sleep(0.3)

    print(f"\nTotal fetched: {len(all_articles)} articles")
    return all_articles


# ── Run ────────────────────────────────────────────────────────────────────
raw_articles = fetch_rss_articles(RSS_SOURCES)

Fetching FIFA News ... (parse warning) OK (0 articles)
Fetching BBC Sport ... OK (85 articles)
Fetching Sky Sports ... OK (20 articles)
Fetching ESPN Soccer ... OK (28 articles)
Fetching Goal.com ... (parse warning) OK (0 articles)
Fetching Reuters Sport ... (parse warning) OK (0 articles)

Total fetched: 133 articles


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 – Deduplication (Jaccard + URL-level)
# ─────────────────────────────────────────────────────────────────

_STOP_WORDS = frozenset({
    "a","an","the","in","on","at","of","to","and","or","but",
    "is","are","was","were","for","with","from","by","as","it",
    "its","this","that","he","she","they","his","her","has",
    "have","had","be","been","will","not",
})


def _tokens(title):
    words = re.sub(r"[^\w\s]", "", title.lower()).split()
    return frozenset(w for w in words if w not in _STOP_WORDS)


def deduplicate_articles(articles, threshold=0.60):
    """
    Remove near-duplicate articles using Jaccard title similarity.
    Also deduplicates by exact URL match.
    """
    unique, seen_token_sets, seen_urls = [], [], set()

    for article in articles:
        link = article.get("link", "")
        if link and link in seen_urls:
            continue
        if link:
            seen_urls.add(link)

        tok = _tokens(article["title"])
        is_dup = any(
            (len(tok & seen) / len(tok | seen)) >= threshold
            for seen in seen_token_sets
            if tok and seen
        )
        if not is_dup:
            unique.append(article)
            seen_token_sets.append(tok)

    removed = len(articles) - len(unique)
    print(f"Dedup: {len(articles)} → {len(unique)} articles ({removed} removed)")
    return unique


# ── Run ────────────────────────────────────────────────────────────────────
unique_articles = deduplicate_articles(raw_articles)

Dedup: 133 → 131 articles (2 removed)


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 – Provider-agnostic AI client
#
# Public API:  ask_ai(prompt, system=None) -> str
# Supports:    groq | claude | gemini
# Switch by changing AI_PROVIDER in .env, then re-running Cell 1.
# ─────────────────────────────────────────────────────────────────


def ask_ai(prompt, system=None, max_tokens=2048):
    """Route to the correct AI backend based on AI_PROVIDER."""
    if AI_PROVIDER == "groq":
        return _ask_groq(prompt, system, max_tokens)
    elif AI_PROVIDER == "claude":
        return _ask_claude(prompt, system, max_tokens)
    elif AI_PROVIDER == "gemini":
        return _ask_gemini(prompt, system, max_tokens)
    else:
        raise ValueError(f"Unknown AI_PROVIDER='{AI_PROVIDER}'")


def _ask_groq(prompt, system, max_tokens):
    from groq import Groq
    client   = Groq(api_key=AI_API_KEY)
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=AI_MODEL, messages=messages,
        max_tokens=max_tokens, temperature=0.7,
    )
    return resp.choices[0].message.content


def _ask_claude(prompt, system, max_tokens):
    import anthropic
    client = anthropic.Anthropic(api_key=AI_API_KEY)
    kwargs = dict(model=AI_MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    msg = client.messages.create(**kwargs)
    return msg.content[0].text


def _ask_gemini(prompt, system, max_tokens):
    import google.generativeai as genai
    genai.configure(api_key=AI_API_KEY)
    full = f"{system}\n\n{prompt}" if system else prompt
    model = genai.GenerativeModel(AI_MODEL)
    resp  = model.generate_content(
        full, generation_config={"max_output_tokens": max_tokens}
    )
    return resp.text


# ── Quick sanity check ─────────────────────────────────────────────────────
print("Testing AI connection ...")
test = ask_ai("Reply with exactly: AI client is working!")
print(f"Response: {test.strip()}")

Testing AI connection ...
Response: AI client is working!


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 – Multi-dimension AI scoring
#
# Scores every article on 4 axes:
#   relevance_score  (0-100) – FIFA 2027 relevance
#   viral_score      (0-100) – social virality
#   breaking_score   (0-100) – time-sensitivity
#   engagement_score (0-100) – comment/share/reaction potential
#
# Also extracts: category, entities (teams, players, countries)
# ─────────────────────────────────────────────────────────────────

BATCH_SIZE = 15
_SCORE_SYSTEM = (
    "You are a FIFA World Cup 2027 social-media strategist. "
    "Respond only with valid JSON."
)
_W = dict(viral=0.35, relevance=0.30, breaking=0.20, engagement=0.15)


def _build_scoring_prompt(batch):
    lines = []
    for i, art in enumerate(batch, start=1):
        line = f"{i}. [{art['source']}] {art['title']}"
        if art.get("summary"):
            line += f"\n   {art['summary'][:150]}"
        lines.append(line)

    return f"""Score these football articles for a FIFA 2027 social-media pipeline.

For EACH article return a JSON object with EXACTLY these fields:
  "index"           : article number (integer, 1-based)
  "relevance_score" : 0-100 (FIFA 2027 / World Cup relevance)
  "viral_score"     : 0-100 (social viral potential)
  "breaking_score"  : 0-100 (how breaking / time-sensitive)
  "engagement_score": 0-100 (comment/share/reaction potential)
  "why"             : one sentence on top viral hook
  "category"        : transfer | injury | result | announcement |
                      squad | controversy | logistics | stadium | other
  "entities"        : object with keys teams[], players[], countries[], stage

Return ONLY a valid JSON array. No markdown. No extra text.

Articles:
{chr(10).join(lines)}"""


def _parse_batch(response_text, batch):
    clean = re.sub(r"```(?:json)?|```", "", response_text).strip().strip("`")
    try:
        scored_list = json.loads(clean)
    except json.JSONDecodeError as exc:
        print(f"  WARNING: JSON parse error ({exc}). Assigning default scores.")
        for art in batch:
            for k, v in [("relevance_score",0),("viral_score",0),("breaking_score",0),
                         ("engagement_score",0),("why",""),("category","other"),("entities",{})]:
                art.setdefault(k, v)
            art["rank_score"] = 0
        return batch

    lookup = {item["index"]: item for item in scored_list if "index" in item}
    for i, art in enumerate(batch, start=1):
        ai = lookup.get(i, {})
        art["relevance_score"]  = int(ai.get("relevance_score",  0))
        art["viral_score"]      = int(ai.get("viral_score",       0))
        art["breaking_score"]   = int(ai.get("breaking_score",    0))
        art["engagement_score"] = int(ai.get("engagement_score",  0))
        art["why"]              = ai.get("why",      "")
        art["category"]         = ai.get("category", "other").lower()
        art["entities"]         = ai.get("entities", {})
        art["rank_score"]       = int(
            art["viral_score"]       * _W["viral"]
            + art["relevance_score"] * _W["relevance"]
            + art["breaking_score"]  * _W["breaking"]
            + art["engagement_score"]* _W["engagement"]
        )
    return batch


def score_articles(articles):
    """Score all articles in batches; return sorted by rank_score desc."""
    all_scored    = []
    n_batches = (len(articles) + BATCH_SIZE - 1) // BATCH_SIZE

    for b in range(n_batches):
        batch = articles[b * BATCH_SIZE : (b+1) * BATCH_SIZE]
        print(f"  Scoring batch {b+1}/{n_batches} ({len(batch)} articles) ...")
        response  = ask_ai(_build_scoring_prompt(batch), system=_SCORE_SYSTEM)
        all_scored.extend(_parse_batch(response, batch))
        if b < n_batches - 1:
            time.sleep(1)

    all_scored.sort(key=lambda a: a.get("rank_score", 0), reverse=True)
    top = all_scored[0]
    print(f"\nTop story (rank={top['rank_score']}): {top['title'][:60]}")
    return all_scored


# ── Run ────────────────────────────────────────────────────────────────────
print("Scoring articles ...\n")
scored_articles = score_articles(unique_articles)
top_articles = [
    a for a in scored_articles
    if a.get("viral_score", 0) >= MIN_VIRAL_SCORE
][:MAX_ARTICLES]
print(f"\n{len(top_articles)} articles passed the viral score threshold.")

Scoring articles ...

  Scoring batch 1/9 (15 articles) ...
  Scoring batch 2/9 (15 articles) ...
  Scoring batch 3/9 (15 articles) ...
  Scoring batch 4/9 (15 articles) ...
  Scoring batch 5/9 (15 articles) ...
  Scoring batch 6/9 (15 articles) ...
  Scoring batch 7/9 (15 articles) ...
  Scoring batch 8/9 (15 articles) ...
  Scoring batch 9/9 (11 articles) ...

Top story (rank=93): Lukaku's instant impact for Belgium's equaliser

20 articles passed the viral score threshold.


In [6]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 – Full content + image-prompt generation
#
# For each top article, generates:
#   • headline / summary  • Facebook caption
#   • Instagram caption   • Twitter/X caption
#   • Hashtags            • Poster text (image overlay)
#   • SEO metadata        • AI image-generation prompt
# ─────────────────────────────────────────────────────────────────

_CONTENT_SYSTEM = (
    "You are an award-winning FIFA World Cup 2027 social-media editor. "
    "Write punchy, platform-optimised content. Respond only with valid JSON."
)

_STYLE_SUFFIX = (
    "professional sports editorial photography, FIFA World Cup 2027 branding, "
    "dynamic stadium lighting, packed crowd atmosphere, 8K ultra-detailed, "
    "sharp focus, photorealistic, high contrast, vivid colors, cinematic"
)

_NEGATIVE_PROMPT = (
    "text, watermark, logo, blurry, low quality, deformed, ugly, "
    "bad anatomy, duplicate, out of frame"
)


def _content_prompt(article):
    ent  = article.get("entities", {})
    return f"""Write complete social-media content for this football story.

Story:
  Title    : {article['title']}
  Source   : {article['source']}
  Summary  : {article.get('summary','N/A')[:250]}
  Category : {article.get('category','other')}
  Hook     : {article.get('why','N/A')}
  Teams    : {', '.join(ent.get('teams',[])  ) or 'N/A'}
  Players  : {', '.join(ent.get('players',[])) or 'N/A'}

Return ONLY a JSON object with EXACTLY these keys:
  "headline"          : punchy headline, max 12 words
  "summary"           : 2-3 factual engaging sentences
  "facebook_caption"  : professional Facebook post, 2-3 paragraphs
  "instagram_caption" : punchy IG caption with line breaks
  "twitter_caption"   : max 240 chars
  "hashtags"          : array of 8-12 hashtags (no # symbol)
  "poster_text"       : ultra-short headline for image overlay, max 7 words
  "image_prompt"      : 100-150 word visual prompt for Stable Diffusion XL
                        (no text/logos/watermarks in the image)
  "negative_prompt"   : what to avoid in the image
  "seo_keywords"      : array of 5-8 keyword phrases
  "meta_description"  : 150-char SEO meta description

No markdown. Valid JSON only."""


def generate_content(article):
    try:
        raw   = ask_ai(_content_prompt(article), system=_CONTENT_SYSTEM)
        clean = re.sub(r"```(?:json)?|```", "", raw).strip().strip("`")
        data  = json.loads(clean)
        article["headline"]          = data.get("headline",         article["title"])
        article["ai_summary"]        = data.get("summary",          article.get("summary",""))
        article["facebook_caption"]  = data.get("facebook_caption", "")
        article["instagram_caption"] = data.get("instagram_caption","")
        article["twitter_caption"]   = data.get("twitter_caption",  "")
        article["hashtags"]          = data.get("hashtags",         [])
        article["poster_text"]       = data.get("poster_text",      article["title"][:40])
        article["image_prompt"]      = data.get("image_prompt",     "") + ", " + _STYLE_SUFFIX
        article["negative_prompt"]   = data.get("negative_prompt",  _NEGATIVE_PROMPT)
        article["seo_keywords"]      = data.get("seo_keywords",     [])
        article["meta_description"]  = data.get("meta_description", "")
    except Exception as exc:
        print(f"    WARNING: content failed for '{article['title'][:40]}': {exc}")
        article.setdefault("headline",         article["title"])
        article.setdefault("ai_summary",       article.get("summary",""))
        article.setdefault("facebook_caption", "")
        article.setdefault("instagram_caption","")
        article.setdefault("twitter_caption",  "")
        article.setdefault("hashtags",         [])
        article.setdefault("poster_text",      article["title"][:40])
        article.setdefault("image_prompt",     "FIFA 2027 football, " + _STYLE_SUFFIX)
        article.setdefault("negative_prompt",  _NEGATIVE_PROMPT)
        article.setdefault("seo_keywords",     [])
        article.setdefault("meta_description", "")
    return article


# ── Run ────────────────────────────────────────────────────────────────────
print("Generating content and image prompts ...\n")
for idx, art in enumerate(top_articles, start=1):
    print(f"  [{idx}/{len(top_articles)}] {art['title'][:55]}")
    generate_content(art)
    time.sleep(0.5)

print(f"\nContent generation complete for {len(top_articles)} articles.")

Generating content and image prompts ...

  [1/20] Lukaku's instant impact for Belgium's equaliser
  [2/20] Spain held by Cape Verde in 1st major WC shock
  [3/20] 'My word!' - Ashour's long-range strikes put Egypt ahea
  [4/20] Japan come from behind twice to draw with Netherlands
  [5/20] 'He's the story' - Vozinha's epic goalkeeping display t
  [6/20] Gyokeres & Isak score as Sweden put five past Tunisia
  [7/20] Cape Verde hold Spain in one of World Cup's biggest sho
  [8/20] Calls for World Cup VAR official to be removed after 'w
  [9/20] Copy of Japan justify dark horse credentials with drama
  [10/20] Bailed out by Vinícius Júnior, Brazil are still a ...
  [11/20] Amad scores 90th-minute winner for Ivory Coast
  [12/20] Amad scores late winner as Ivory Coast beat Ecuador
  [13/20] Havertz scores twice as Germany thrash Curacao
  [14/20] The 40-year-old keeper who inspired Cape Verde's histor
  [15/20] Vozinha heroics help Cape Verde hold Spain to goalless 
  [16/20] Emotional fu

In [7]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 – Background image generation
#
# Generates one background image per top article.
# Backend is controlled by IMAGE_BACKEND in .env:
#   mock       – solid dark-blue placeholder (no API needed)
#   stability  – Stability AI (STABILITY_API_KEY required)
#   replicate  – Flux via Replicate (REPLICATE_API_KEY required)
#
# Each image is saved to the posters/ directory.
# ─────────────────────────────────────────────────────────────────

import os
from pathlib import Path

POSTERS_DIR = Path("posters")
POSTERS_DIR.mkdir(exist_ok=True)

STABILITY_API_KEY = os.getenv("STABILITY_API_KEY", "")
REPLICATE_API_KEY = os.getenv("REPLICATE_API_KEY", "")


def _mock_image(w, h, label="FIFA 2027"):
    """Create a dark-blue placeholder PNG."""
    img  = Image.new("RGB", (w, h), (8, 20, 50))
    draw = ImageDraw.Draw(img)
    for y in range(h):
        shade = int(30 + 20 * y / h)
        draw.line([(0, y), (w, y)], fill=(shade, shade * 2, shade * 5))
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 32
        )
    except Exception:
        font = ImageFont.load_default()
    draw.text((w // 2, h // 2), f"FIFA 2027 | {label}", fill=(212,175,55),
              anchor="mm", font=font)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    return buf


def _stability_image(prompt, negative_prompt, w, h):
    import requests as req
    resp = req.post(
        "https://api.stability.ai/v2beta/stable-image/generate/sd3",
        headers={"authorization": f"Bearer {STABILITY_API_KEY}", "accept": "image/*"},
        files={"none": ""},
        data={"prompt": prompt, "negative_prompt": negative_prompt,
              "width": str(w), "height": str(h), "output_format": "png"},
        timeout=120,
    )
    resp.raise_for_status()
    return io.BytesIO(resp.content)


def _replicate_image(prompt, negative_prompt, w, h):
    import replicate, requests as req
    output = replicate.run(
        "black-forest-labs/flux-1.1-pro",
        input={"prompt": prompt, "negative_prompt": negative_prompt,
               "width": w, "height": h, "output_format": "png"},
    )
    url  = str(output)
    resp = req.get(url, timeout=120)
    resp.raise_for_status()
    return io.BytesIO(resp.content)


def generate_bg_image(article, size=(1080, 1080)):
    w, h = size
    prompt    = article.get("image_prompt", "FIFA 2027 football match")
    neg       = article.get("negative_prompt", "text, watermark, blurry")
    slug      = "".join(c if c.isalnum() else "_" for c in article["title"][:30])
    out_path  = POSTERS_DIR / f"bg_{slug}.png"

    try:
        if IMAGE_BACKEND == "stability" and STABILITY_API_KEY:
            buf = _stability_image(prompt, neg, w, h)
        elif IMAGE_BACKEND == "replicate" and REPLICATE_API_KEY:
            buf = _replicate_image(prompt, neg, w, h)
        else:
            buf = _mock_image(w, h)

        img = Image.open(buf).convert("RGB").resize((w, h))
        img.save(out_path, "PNG")
        return out_path

    except Exception as exc:
        print(f"    Image gen failed ({exc}); using mock.")
        buf = _mock_image(w, h)
        img = Image.open(buf)
        img.save(out_path, "PNG")
        return out_path


# ── Run ────────────────────────────────────────────────────────────────────
print(f"Generating background images  backend={IMAGE_BACKEND} ...\n")
for idx, art in enumerate(top_articles, start=1):
    print(f"  [{idx}/{len(top_articles)}] {art['title'][:50]}")
    art["bg_image_path"] = str(generate_bg_image(art))

print(f"\nImages saved to {POSTERS_DIR}/")

Generating background images  backend=gemini ...

  [1/20] Lukaku's instant impact for Belgium's equaliser
  [2/20] Spain held by Cape Verde in 1st major WC shock
  [3/20] 'My word!' - Ashour's long-range strikes put Egypt
  [4/20] Japan come from behind twice to draw with Netherla
  [5/20] 'He's the story' - Vozinha's epic goalkeeping disp
  [6/20] Gyokeres & Isak score as Sweden put five past Tuni
  [7/20] Cape Verde hold Spain in one of World Cup's bigges
  [8/20] Calls for World Cup VAR official to be removed aft
  [9/20] Copy of Japan justify dark horse credentials with 
  [10/20] Bailed out by Vinícius Júnior, Brazil are still a 
  [11/20] Amad scores 90th-minute winner for Ivory Coast
  [12/20] Amad scores late winner as Ivory Coast beat Ecuado
  [13/20] Havertz scores twice as Germany thrash Curacao
  [14/20] The 40-year-old keeper who inspired Cape Verde's h
  [15/20] Vozinha heroics help Cape Verde hold Spain to goal
  [16/20] Emotional full-time scenes as Cape Verde celebrat

In [16]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 – Background image generation
#
# Generates one background image per top article.
# Backend is controlled by IMAGE_BACKEND in .env:
#   mock       – solid dark-blue placeholder (no API needed)
#   stability  – Stability AI (STABILITY_API_KEY required)
#   replicate  – Flux via Replicate (REPLICATE_API_KEY required)
#
# Each image is saved to the posters/ directory.
# ─────────────────────────────────────────────────────────────────

import os
from pathlib import Path

POSTERS_DIR = Path("posters")
POSTERS_DIR.mkdir(exist_ok=True)

STABILITY_API_KEY = os.getenv("STABILITY_API_KEY", "")
REPLICATE_API_KEY = os.getenv("REPLICATE_API_KEY", "")


def _mock_image(w, h, label="FIFA 2027"):
    """Create a dark-blue placeholder PNG."""
    img  = Image.new("RGB", (w, h), (8, 20, 50))
    draw = ImageDraw.Draw(img)
    for y in range(h):
        shade = int(30 + 20 * y / h)
        draw.line([(0, y), (w, y)], fill=(shade, shade * 2, shade * 5))
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 32
        )
    except Exception:
        font = ImageFont.load_default()
    draw.text((w // 2, h // 2), f"FIFA 2027 | {label}", fill=(212,175,55),
              anchor="mm", font=font)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    return buf


def _stability_image(prompt, negative_prompt, w, h):
    import requests as req
    resp = req.post(
        "https://api.stability.ai/v2beta/stable-image/generate/sd3",
        headers={"authorization": f"Bearer {STABILITY_API_KEY}", "accept": "image/*"},
        files={"none": ""},
        data={"prompt": prompt, "negative_prompt": negative_prompt,
              "width": str(w), "height": str(h), "output_format": "png"},
        timeout=120,
    )
    resp.raise_for_status()
    return io.BytesIO(resp.content)


def _replicate_image(prompt, negative_prompt, w, h):
    import replicate, requests as req
    output = replicate.run(
        "black-forest-labs/flux-1.1-pro",
        input={"prompt": prompt, "negative_prompt": negative_prompt,
               "width": w, "height": h, "output_format": "png"},
    )
    url  = str(output)
    resp = req.get(url, timeout=120)
    resp.raise_for_status()
    return io.BytesIO(resp.content)


def generate_bg_image(article, size=(1080, 1080)):
    w, h = size
    prompt    = article.get("image_prompt", "FIFA 2027 football match")
    neg       = article.get("negative_prompt", "text, watermark, blurry")
    slug      = "".join(c if c.isalnum() else "_" for c in article["title"][:30])
    out_path  = POSTERS_DIR / f"bg_{slug}.png"

    try:
        if IMAGE_BACKEND == "stability" and STABILITY_API_KEY:
            buf = _stability_image(prompt, neg, w, h)
        elif IMAGE_BACKEND == "replicate" and REPLICATE_API_KEY:
            buf = _replicate_image(prompt, neg, w, h)
        else:
            buf = _mock_image(w, h)

        img = Image.open(buf).convert("RGB").resize((w, h))
        img.save(out_path, "PNG")
        return out_path

    except Exception as exc:
        print(f"    Image gen failed ({exc}); using mock.")
        buf = _mock_image(w, h)
        img = Image.open(buf)
        img.save(out_path, "PNG")
        return out_path


# ── Run ────────────────────────────────────────────────────────────────────
print(f"Generating background images  backend={IMAGE_BACKEND} ...\n")
for idx, art in enumerate(top_articles, start=1):
    print(f"  [{idx}/{len(top_articles)}] {art['title'][:50]}")
    art["bg_image_path"] = str(generate_bg_image(art))

print(f"\nImages saved to {POSTERS_DIR}/")

Generating background images  backend=gemini ...

  [1/20] Lukaku's instant impact for Belgium's equaliser
  [2/20] Spain held by Cape Verde in 1st major WC shock
  [3/20] 'My word!' - Ashour's long-range strikes put Egypt
  [4/20] Japan come from behind twice to draw with Netherla
  [5/20] 'He's the story' - Vozinha's epic goalkeeping disp
  [6/20] Gyokeres & Isak score as Sweden put five past Tuni
  [7/20] Cape Verde hold Spain in one of World Cup's bigges
  [8/20] Calls for World Cup VAR official to be removed aft
  [9/20] Copy of Japan justify dark horse credentials with 
  [10/20] Bailed out by Vinícius Júnior, Brazil are still a 
  [11/20] Amad scores 90th-minute winner for Ivory Coast
  [12/20] Amad scores late winner as Ivory Coast beat Ecuado
  [13/20] Havertz scores twice as Germany thrash Curacao
  [14/20] The 40-year-old keeper who inspired Cape Verde's h
  [15/20] Vozinha heroics help Cape Verde hold Spain to goal
  [16/20] Emotional full-time scenes as Cape Verde celebrat

In [18]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 – Background image generation
#
# Generates one background image per top article.
# Backend is controlled by IMAGE_BACKEND in .env:
#   mock       – solid dark-blue placeholder (no API needed)
#   stability  – Stability AI (STABILITY_API_KEY required)
#   replicate  – Flux via Replicate (REPLICATE_API_KEY required)
#
# Each image is saved to the posters/ directory.
# ─────────────────────────────────────────────────────────────────

import os
from pathlib import Path

POSTERS_DIR = Path("posters")
POSTERS_DIR.mkdir(exist_ok=True)

STABILITY_API_KEY = os.getenv("STABILITY_API_KEY", "")
REPLICATE_API_KEY = os.getenv("REPLICATE_API_KEY", "")
GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY", "")


def _mock_image(w, h, label="FIFA 2027"):
    """Create a dark-blue placeholder PNG."""
    img  = Image.new("RGB", (w, h), (8, 20, 50))
    draw = ImageDraw.Draw(img)
    for y in range(h):
        shade = int(30 + 20 * y / h)
        draw.line([(0, y), (w, y)], fill=(shade, shade * 2, shade * 5))
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 32
        )
    except Exception:
        font = ImageFont.load_default()
    draw.text((w // 2, h // 2), f"FIFA 2027 | {label}", fill=(212,175,55),
              anchor="mm", font=font)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    return buf


def _stability_image(prompt, negative_prompt, w, h):
    import requests as req
    resp = req.post(
        "https://api.stability.ai/v2beta/stable-image/generate/sd3",
        headers={"authorization": f"Bearer {STABILITY_API_KEY}", "accept": "image/*"},
        files={"none": ""},
        data={"prompt": prompt, "negative_prompt": negative_prompt,
              "width": str(w), "height": str(h), "output_format": "png"},
        timeout=120,
    )
    resp.raise_for_status()
    return io.BytesIO(resp.content)


def _replicate_image(prompt, negative_prompt, w, h):
    import replicate, requests as req
    output = replicate.run(
        "black-forest-labs/flux-1.1-pro",
        input={"prompt": prompt, "negative_prompt": negative_prompt,
               "width": w, "height": h, "output_format": "png"},
    )
    url  = str(output)
    resp = req.get(url, timeout=120)
    resp.raise_for_status()
    return io.BytesIO(resp.content)


def _gemini_image(prompt, w, h):
    import google.generativeai as genai
    import requests as req

    genai.configure(api_key=GEMINI_API_KEY)

    model = genai.ImageGenerationModel(
        "imagen-3.0-generate-002"
    )

    response = model.generate_images(
        prompt=prompt,
        number_of_images=1,
        aspect_ratio="1:1" if w == h else (
            "9:16" if h > w else "16:9"
        ),
    )

    img = response.images[0]

    buf = io.BytesIO()
    img._pil_image.save(buf, format="PNG")
    buf.seek(0)

    return buf


def generate_bg_image(article, size=(1080, 1080)):
    w, h = size
    prompt    = article.get("image_prompt", "FIFA 2027 football match")
    neg       = article.get("negative_prompt", "text, watermark, blurry")
    slug      = "".join(c if c.isalnum() else "_" for c in article["title"][:30])
    out_path  = POSTERS_DIR / f"bg_{slug}.png"

    try:
        if IMAGE_BACKEND == "stability" and STABILITY_API_KEY:
            buf = _stability_image(prompt, neg, w, h)
        elif IMAGE_BACKEND == "replicate" and REPLICATE_API_KEY:
            buf = _replicate_image(prompt, neg, w, h)
        else:
            buf = _mock_image(w, h)

        img = Image.open(buf).convert("RGB").resize((w, h))
        img.save(out_path, "PNG")
        return out_path

    except Exception as exc:
        print(f"    Image gen failed ({exc}); using mock.")
        buf = _mock_image(w, h)
        img = Image.open(buf)
        img.save(out_path, "PNG")
        return out_path


# ── Run ────────────────────────────────────────────────────────────────────
print(f"Generating background images  backend={IMAGE_BACKEND} ...\n")
for idx, art in enumerate(top_articles, start=1):
    print(f"  [{idx}/{len(top_articles)}] {art['title'][:50]}")
    art["bg_image_path"] = str(generate_bg_image(art))

print(f"\nImages saved to {POSTERS_DIR}/")

Generating background images  backend=gemini ...

  [1/20] Lukaku's instant impact for Belgium's equaliser
  [2/20] Spain held by Cape Verde in 1st major WC shock
  [3/20] 'My word!' - Ashour's long-range strikes put Egypt
  [4/20] Japan come from behind twice to draw with Netherla
  [5/20] 'He's the story' - Vozinha's epic goalkeeping disp
  [6/20] Gyokeres & Isak score as Sweden put five past Tuni
  [7/20] Cape Verde hold Spain in one of World Cup's bigges
  [8/20] Calls for World Cup VAR official to be removed aft
  [9/20] Copy of Japan justify dark horse credentials with 
  [10/20] Bailed out by Vinícius Júnior, Brazil are still a 
  [11/20] Amad scores 90th-minute winner for Ivory Coast
  [12/20] Amad scores late winner as Ivory Coast beat Ecuado
  [13/20] Havertz scores twice as Germany thrash Curacao
  [14/20] The 40-year-old keeper who inspired Cape Verde's h
  [15/20] Vozinha heroics help Cape Verde hold Spain to goal
  [16/20] Emotional full-time scenes as Cape Verde celebrat